## 03．データ分割

１．計算した分子記述子と温度・湿度などの環境変数・配合情報を合体

２．WVPの値を対数変換（log1p）

３．RDKitの欠損値を完全に排除した上で、訓練用（80%）と検証用（20%）に分割

②experiment_idでくくる

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# データの読み込み
df_clean = pd.read_csv(r"C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\interim\260625\wvp_adjustment\01_cleaned_raw.csv")
df_desc = pd.read_csv(r"C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\interim\260625\wvp_adjustment\02_molecular_descriptors.csv")

def prepare_dataset_grouped(df_clean, df_desc):
    print("--- Step 3: 特徴量結合・対数変換・グループデータ分割 ---")
    
    # 1. 計算した分子記述子と環境変数・配合情報を合体
    # ※ 分割の基準にするため、あえて 'experiment_id' も一緒に結合しておきます
    X_raw = pd.concat([
        df_clean[['experiment_id', 'humidity', 'temperature', 'material_concentration', 'proportion']].reset_index(drop=True), 
        df_desc
    ], axis=1)
    
    y_raw = df_clean['wvp'].reset_index(drop=True)
    
    # 3. RDKitの欠損値を完全に排除
    valid_idx = X_raw.notna().all(axis=1)
    X_valid = X_raw[valid_idx].copy()
    
    # 2. WVPの値を対数変換（log1p）
    y_valid = np.log1p(y_raw[valid_idx])
    
    # 【超重要】グループ分けの基準となる列を抽出して、特徴量からは削除する
    groups = X_valid['experiment_id'].values
    X_final = X_valid.drop(columns=['experiment_id'])
    
    # ─── 💡 GroupShuffleSplit によるグループ保護分割 ───
    # test_size=0.2 (20%)、同じ experiment_id は必ず同じ側にグループ分けされる
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    
    # groups を指定してインデックス（行番号）を取得
    train_idx, test_idx = next(gss.split(X_final, y_valid, groups=groups))
    
    # インデックスを元にデータを訓練用と検証用に分割
    X_train, X_test = X_final.iloc[train_idx], X_final.iloc[test_idx]
    y_train, y_test = y_valid.iloc[train_idx], y_valid.iloc[test_idx]
    
    # ─── 🧪 検証用のログ（本当に分かれているかチェック） ───
    train_groups = set(groups[train_idx])
    test_groups = set(groups[test_idx])
    overlap = train_groups.intersection(test_groups)
    
    print(f"訓練データの固有グループ数: {len(train_groups)}")
    print(f"テストデータの固有グループ数: {len(test_groups)}")
    print(f"重複しているグループ数: {len(overlap)} (0なら大成功！)")
    print("-" * 40)
    
    return X_train, X_test, y_train, y_test

if __name__ == "__main__":
    # 関数の実行
    X_train, X_test, y_train, y_test = prepare_dataset_grouped(df_clean, df_desc)
    
    print(f"訓練データ（特徴量）の形状: {X_train.shape}")
    print(f"テストデータ（特徴量）の形状: {X_test.shape}")
    
    # データの保存
    export_path = r"C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\interim\260625\wvp_adjustment\03_2_split_dataset.npz"
    np.savez(export_path, 
             X_train=X_train.values, 
             X_test=X_test.values, 
             y_train=y_train.values, 
             y_test=y_test.values,
             columns=X_train.columns.tolist())
    
    print("Step 3 のグループ分割データを正常に保存しました！")

--- Step 3: 特徴量結合・対数変換・グループデータ分割 ---
訓練データの固有グループ数: 53
テストデータの固有グループ数: 14
重複しているグループ数: 0 (0なら大成功！)
----------------------------------------
訓練データ（特徴量）の形状: (3213, 25)
テストデータ（特徴量）の形状: (877, 25)
Step 3 のグループ分割データを正常に保存しました！
